In [1]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
model.info(verbose=True)  # detailed layer info


YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs


(129, 11166560, 0, 28.816844800000002)

In [7]:
from ultralytics import YOLO
from torchinfo import summary

# Load YOLOv8s (small) or your custom model
model = YOLO("yolov8s.pt").model  # .model gives the underlying PyTorch nn.Module

# Print architecture summary
summary(model, input_size=(1, 3, 640, 640))


Layer (type:depth-idx)                             Output Shape              Param #
DetectionModel                                     [1, 84, 8400]             --
├─Sequential: 1-1                                  --                        --
│    └─Conv: 2-1                                   [1, 32, 320, 320]         --
│    │    └─Conv2d: 3-1                            [1, 32, 320, 320]         (864)
│    │    └─BatchNorm2d: 3-2                       [1, 32, 320, 320]         (64)
│    └─Detect: 2-96                                --                        (recursive)
│    │    └─ModuleList: 3-118                      --                        (recursive)
│    └─Conv: 2-3                                   [1, 64, 160, 160]         --
│    │    └─Conv2d: 3-4                            [1, 64, 160, 160]         (18,432)
│    │    └─BatchNorm2d: 3-5                       [1, 64, 160, 160]         (128)
│    └─Detect: 2-96                                --                        (recur

In [10]:
import torch
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os
from PIL import Image
import torchvision.transforms as T

# ----------------------
# Load YOLOv8 model
# ----------------------
model = YOLO("D:/Pill_Identification/model/YOLOv8/runs/detect/train7/weights/best.pt")  # your trained model
model = model.model  # get torch.nn.Module

# ----------------------
# Prepare input image
# ----------------------
img_path = "D:/Pill_Identification/dataset/Pill_jpg_2025_Original/1/MT/Mau/1_MT_01.jpg"  # <-- change to your pill image
img = Image.open(img_path).convert("RGB")

transform = T.Compose([
    T.Resize((640, 640)),
    T.ToTensor(),   # converts to [0,1] tensor
])
x = transform(img).unsqueeze(0)  # [1,3,640,640]

# ----------------------
# Hook system to capture outputs
# ----------------------
layer_outputs = {}

def hook_fn(module, input, output):
    layer_outputs[module] = output

# Register hooks for leaf layers
for name, layer in model.named_modules():
    if not list(layer.children()):
        layer.register_forward_hook(hook_fn)

# Forward pass
with torch.no_grad():
    _ = model(x)

print(f"Captured {len(layer_outputs)} layers")

# ----------------------
# Save feature maps
# ----------------------
save_dir = "yolo_pill_feature_maps"
os.makedirs(save_dir, exist_ok=True)

def save_feature_maps(feats, layer_name, max_channels=4):
    feats = feats[0]  # first image from batch
    C = feats.shape[0]

    for i in range(min(max_channels, C)):
        plt.imshow(feats[i].cpu().numpy(), cmap="viridis")
        plt.axis("off")
        plt.title(f"{layer_name} - ch{i}")
        fname = os.path.join(save_dir, f"{layer_name}_ch{i}.png")
        plt.savefig(fname, bbox_inches="tight", pad_inches=0.1)
        plt.close()

# Loop and save
for i, (layer, out) in enumerate(layer_outputs.items()):
    layer_name = f"{i}_{layer.__class__.__name__}"
    save_feature_maps(out, layer_name, max_channels=4)

print(f"✅ Feature maps saved in: {save_dir}")


Captured 129 layers
✅ Feature maps saved in: yolo_pill_feature_maps
